<h1 align="center">TBR Analysis (Water - Graphene Oxide)</h1>


**Author:** F. Tarulli, Politecnico di Torino, Italy - Jun 7, 2025 (UPDATE: Nov 25, 2025)

**Co-Author** P. De Angelis, Politecnico di Torino, Italy

<br>

Script loads transient LAMMPS outputs, computes ∫ΔT*dt, finds linear regime,  
fits OLS to extract Kapitza resistance ($\mathrm{R_K}$) and conductance ($\mathrm{G_K}$),  
and plots results. <br>It allows to processes multiple simulations (to achieve robust estimates of the TBR, the analysis should be repeated over an ensemble of independent runs).

**Adjust “Variables” section before running:**

- `TBulk`  
  *If True, reads water bulk temperature; if False, uses slab-based temperature profile*  
- `h_slab`  
  *Slab half-thickness in Å for averaging water temperature around the graphene*  
- `oxid`  
  *Oxidation degree (%) for the –OH coverage on the graphene sheet*  
- `replicas`
  *Number of replicas of indipendent runs*

<br>

**IMPORTANT**: adapt `base_remote_path` to handle multiple repetitions in `Load` section. Note that with 1 replica the fitting procedure could fail.


# Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.integrate import cumulative_trapezoid
import os
from scipy.constants import N_A
from tqdm.auto import tqdm
from scipy.optimize import minimize_scalar
import statsmodels.api as sm
from time import time
import scipy.stats

In [ ]:
def read_Tgraph_chunks(base_remote_path, file_name, k):
    Temp_chunks = []
    with open(f'{base_remote_path}/slab_position_check.txt', 'r') as file:
        for _ in range(3):
            next(file)
        line = next(file).strip().split()
        Nchunks_tot = int(line[1])
    with open(f'{base_remote_path}/{file_name}', 'r') as file:
        for i in range(k):
            if i == 0:
                # Skip the first 4 lines for the first chunk
                for _ in range(4):
                    next(file)
            else:
                # Skip the first 2 lines for subsequent chunks
                next(file)
            
            # Read the next N lines
            chunk = []
            for _ in range(Nchunks_tot):
                line = next(file).strip()
                values = list(map(float, line.split()))
                chunk.append(values)
            
            Temp_chunks.append(np.array(chunk))
    return Temp_chunks, Nchunks_tot

def process_chunks(Temp_chunks, Nchunks_tot, Nchunks, slab_skip=3): 
    k = len(Temp_chunks)
    T_slab_positive = np.zeros(k)
    T_slab_negative = np.zeros(k)
    T_bulk_chunk = np.zeros(k)

    for i in range(k):
        up = Nchunks_tot // 2 + int(slab_skip) 
        down = Nchunks_tot // 2 - 1 - int(slab_skip) 
        
        # Average layers above graphene
        positive_layers = [Temp_chunks[i][up + j, 1] for j in range(Nchunks-slab_skip)]
        T_slab_positive[i] = np.mean(positive_layers)
        
        # Average layers below graphene
        negative_layers = [Temp_chunks[i][down - j, 1] for j in range(Nchunks-slab_skip)]
        T_slab_negative[i] = np.mean(negative_layers)

        # Average temperature across the entire chunk
        T_bulk_chunk[i] = np.mean(Temp_chunks[i][:, 1])
    
    # Mean temperature of both slabs
    T_slab_mean = np.mean(np.vstack((T_slab_positive, T_slab_negative)), axis=0)
    return T_slab_positive, T_slab_negative, T_bulk_chunk, T_slab_mean


def find_breakpoint_iterative(x, y, min_points=10, maxiter=100, penalty_npoint=0.1, penalty_const=0.1):
    def mse_break(x0):
        idx = np.searchsorted(x, x0, 'right')
        c0 = np.mean(y[:10])
        if idx < min_points:
            idx = min_points
            return np.float64(1e100)
        xi, yi = x[:idx], y[:idx]
        yl = y[idx:]
        m, c = np.polyfit(xi, yi, 1)
        const_par = m*x0 + c0
        return np.sum((yi - (m*xi + c0))**2) + penalty_const*np.sum((yl - const_par)**2)  + 0*np.abs(np.log(len(x)/len(xi)))*penalty_npoint

    res = minimize_scalar(
        mse_break,
        bounds=(x[min_points], x[-1]),
        method='bounded',
        options={'maxiter': maxiter}
    )
    return res.x

# Variables

In [ ]:
TBulk = False    # If True, reads water bulk temperature; if False, uses slab-based temperature profile
h_slab = 9       # Slab half-thickness in Å for averaging water temperature around the graphene
oxid = 20        # Oxidation degree (%)
WINDOW = 50      # Rolling window of moving average 
replicas = 1     # Number of indipendent runs 

path_root = Path('../../lammps/TBR/transient')

# Load data

In [ ]:
DATA_ALL = {}
temps_w = []
temps_g = []
pe_g = []
ke_g = []
etot = []
c_rep = 0

print(f" OXIDATION DEGREE = {oxid:3d}% ".center(100, "="))

for rep in tqdm(range(1,replicas+1), desc='Loading replica', total=replicas):
    base_remote_path = Path.joinpath(path_root)
    try:
        file_liquid = 'system_h2o.txt'
        file_nanoparticles = 'system_graph.txt'
        file_chunk = 'temp_chunk_bias_1A.out'
        file_Pot = 'potential_graph.txt'
        file_Kin = 'kinetic_graph.txt'
        temps_g.append(pd.read_csv(os.path.join(base_remote_path, file_nanoparticles), sep=r'\s+',  skiprows=2, names=["step", f"temp_{c_rep:d}"]))
        if TBulk:
            temps_w.append(pd.read_csv(os.path.join(base_remote_path, file_liquid), sep=r'\s+', skiprows=2, names=["step", f"temp_{c_rep:d}"]))
        else: 
            Tchunks, tot_chunks = read_Tgraph_chunks(base_remote_path, file_chunk, temps_g[0].shape[0])
            T_slab_positive, T_slab_negative, T_bulk_chunk, T_slab_mean = process_chunks(Tchunks,tot_chunks,h_slab)
            df_slab = pd.DataFrame({"step": temps_g[-1]["step"].values, f"temp_{c_rep:d}": T_slab_mean })
            temps_w.append(df_slab)
        pe_g.append(pd.read_csv(os.path.join(base_remote_path, file_Pot), sep=r'\s+', skiprows=2, names=["step", f"pe_{c_rep:d}"]))
        ke_g.append(pd.read_csv(os.path.join(base_remote_path, file_Kin), sep=r'\s+', skiprows=2, names=["step", f"ke_{c_rep:d}"]))  
        etot.append( pd.DataFrame(data={"step": pe_g[-1]["step"], f"etot_{c_rep:d}": pe_g[-1][f"pe_{c_rep:d}"] + ke_g[-1][f"ke_{c_rep:d}"]}) )
        c_rep +=1
    except Exception:
        print(f"Error, files of replica ${rep} missing\n")

    rep_Tgr = pd.concat(temps_g, axis=1)
    rep_etot = pd.concat(etot, axis=1)

    rep_Tw = pd.concat(temps_w, axis=1)

    DATA_ALL[oxid] = {
        "Tw": rep_Tw.copy(),
        "Tgr": rep_Tgr.copy(),
        "etot": rep_etot.copy(),
    }

# Concatenate

In [ ]:
CONCAT_ALL_DATA = {}

for oxid, data in tqdm(DATA_ALL.items(),desc="concat data", total=len(DATA_ALL)):
    CONCAT_ALL_DATA[oxid] = {
        "time_s": data["Tw"].iloc[:,0].values * 1e-15, 
        "Tw": np.column_stack([data["Tw"][f"temp_{i}"].values for i in range(replicas)]),
        "Tgr": np.column_stack([data["Tgr"][f"temp_{i}"].values for i in range(replicas)]),
        "etot": np.column_stack([data["etot"][f"etot_{i}"].values for i in range(replicas)]),
    }
    Nt = CONCAT_ALL_DATA[oxid]["time_s"].shape[0]
    int_t_rep = np.zeros((Nt, replicas))
    for i in range(replicas):
        dt_i = data["Tgr"].loc[:, f"temp_{i}"].values - data["Tw"].loc[:, f"temp_{i}"].values
        dt_i = pd.Series(dt_i).rolling(WINDOW).mean().fillna(0.0).values
        int_t_rep[1:, i] = cumulative_trapezoid(dt_i, CONCAT_ALL_DATA[oxid]["time_s"])
    CONCAT_ALL_DATA[oxid]['integral'] = int_t_rep - int_t_rep[5000]

# $R_K$ and $G_K$

## Area

In [ ]:
AREA = {}
try:
    with open(f'{base_remote_path}/water-graph_transient.dump', 'r') as file:
        for _ in range(5):
            next(file)
        x_line = next(file).strip().split()
        y_line = next(file).strip().split()
    xx = float(x_line[1]) - float(x_line[0]) 
    yy = float(y_line[1]) - float(y_line[0])
    AREA[oxid] = 2 * xx*1e-10 * yy*1e-10
except:
    print("Trajectory file not found.\nImport AREA variables (length and width) manually in this cell.\n")
    length =  None  # (m^2) insert value if trajectory file is not available
    width  =  None  # (m^2) insert value if trajectory file is not available
    AREA[oxid] = 2 * length * width

## Slope and plot

In [ ]:
SLOPE_ALL = {}
SLOPE_ensavg_ALL = {}

for oxid, data in CONCAT_ALL_DATA.items():
    start_s = time()
    print(f" OXIDATION DEGREE: {oxid:3d}% ".center(120, "="))
    diffT = pd.DataFrame(data['Tgr'] - data['Tw']).rolling(WINDOW).mean().values[5000:,:]
    integral = data['integral'][5000:]
    etot = data['etot'][5000:]
    scaleE = etot.std()
    scaleInt = integral.std()
    N_each = integral.shape[0] // replicas
    t0_all = []
    m_all = []
    for i in range(replicas):
        t0 = find_breakpoint_iterative(integral[:,i]/scaleInt, etot[:,i]/scaleE, min_points=9, maxiter=500, penalty_npoint=2, penalty_const=1)
        integ_temp = integral[:,i]/scaleInt
        if len(integ_temp[integ_temp<t0]) < 10:
            continue
        t0_all.append(t0*scaleInt)
    t0_min = np.min(t0_all)

    # Merge data from all replicas and compute fitting    
    x_fit = integral[integral<t0_min]
    y_fit = etot[integral<t0_min]
    x_fit = sm.add_constant(x_fit) 
    robust_model = sm.OLS(y_fit, x_fit)
    results = robust_model.fit()
    slope = results.params[1]

    # Indipendent fitting of each replicas
    all_fit = []
    slope_reps = []
    for i in range(replicas):
        integral_rep = integral[:,i]
        etot_rep = etot[:,i]
        t0_i = t0_all[i] 
        x_fit = integral_rep[integral_rep<t0_min]
        y_fit = etot_rep[integral_rep<t0_min]
        x_fit = sm.add_constant(x_fit)
        robust_model = sm.OLS(y_fit, x_fit)
        results_rep = robust_model.fit()
        all_fit.append(
            results_rep
        )
        slope_reps.append( results_rep.params[1] )

    intercept_reps = [res.params[0] for res in all_fit]
    slope_reps     = [res.params[1] for res in all_fit]
    a_bar = np.mean(intercept_reps)
    m_bar = np.mean(slope_reps)
    x_pred = np.linspace(integral.min(), t0_min, 100)
    y_med = a_bar + m_bar * x_pred
    plt.plot(x_pred, y_med, color='r', zorder =20 , lw=4, label="Ensemble-average OLS fit")

    slope = np.mean(slope_reps)
    slope_se   = np.std(slope_reps, ddof=1) / np.sqrt(replicas)
    q_t = scipy.stats.t.ppf(0.975, replicas - 1)                  #t-student
    slope_err  = q_t * slope_se

    SLOPE_ALL[oxid] = {
        "Rk_m" : - AREA[oxid]/ (slope* 4184 / N_A),
        "Rk_e" : AREA[oxid]/ (4184 / N_A) / (slope)**2 * slope_err,
    }

    SLOPE_ALL[oxid]["Gk_m"] = 1/SLOPE_ALL[oxid]["Rk_m"]
    SLOPE_ALL[oxid]["Gk_e"] = SLOPE_ALL[oxid]["Rk_e"] / SLOPE_ALL[oxid]["Rk_m"]**2
    if replicas > 1:
        print(fr"Rk = {SLOPE_ALL[oxid]["Rk_m"]:1.3e} +/- {SLOPE_ALL[oxid]["Rk_e"]:1.3e} (m^2*K/W)")
        print(fr"Gk = {SLOPE_ALL[oxid]["Gk_m"]:1.3e} +/- {SLOPE_ALL[oxid]["Gk_e"]:1.3e} (W/m^2/K)")
        print("")
    else:
        print(fr"Rk = {SLOPE_ALL[oxid]["Rk_m"]:1.3e} (m^2*K/W)")
        print(fr"Gk = {SLOPE_ALL[oxid]["Gk_m"]:1.3e} (W/m^2/K)")
        print("")

    b = results.params
    x_pred = np.linspace(integral.min(), t0_min, replicas)
    pred = results.get_prediction(sm.add_constant(x_pred))
    y_pred = pred.predicted
    iv_l_ols = pred.summary_frame()["mean_ci_lower"]
    iv_u_ols = pred.summary_frame()["mean_ci_upper"]

    n=0
    for res_fint in all_fit:
        indx0 = integral[:, n] < t0_min
        pred = res_fint.get_prediction(sm.add_constant(x_pred))
        y_pred_i = pred.predicted
        if n==0:
            plt.plot(x_pred,y_pred_i, color='k', ls=":", label="Individual OLS fit")
        else:
            plt.plot(x_pred,y_pred_i, color='k', ls=":")
        plt.scatter(integral[indx0,n], etot[indx0,n], s=4, alpha=.3, label = f'Replica {n}')
        n += 1
    plt.xlabel(r"$\int \Delta T\,dt\;(\mathrm{K\cdot s})$", fontsize=12)
    plt.ylabel("Energy (kcal/mol)", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()
    end_time =  time()
    print(f"elalpesed time {end_time-start_s} s")